# Core Training and Rule-Fitting Logic

This notebook keeps the executable parts of the pipeline: model setup, training, checkpoint loading, train-derived OOD cache construction, and train-derived rule fitting. It omits result reporting, sample-specific correctness claims, and submission-feedback analysis.

## Setup: imports and seed

Imports the libraries used by the pipeline and fixes random seeds for Python, NumPy, and PyTorch so repeated runs follow the same intended random sequence.


In [ ]:
%pip install -q -r requirements.txt 2>&1 | tail -3

In [2]:
import copy
import hashlib
import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from tqdm import tqdm
import cv2

# Phase B (V3 ConvNeXt-T) 용
import timm

# Phase E-G (override rule fits) 용
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, classification_report
from skimage.feature import local_binary_pattern

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

In [3]:
# CLAUDE.md 0.4.3 — SEED + env transparency block
print("━" * 60)
print("  🔒 RNG seed configuration")
print("━" * 60)
print(f"  SEED                       : {SEED}")
print(f"  PYTHONHASHSEED             : {os.environ.get('PYTHONHASHSEED', '(not set)')}")
print(f"  random.seed                : {SEED}")
print(f"  np.random.seed             : {SEED}")
print(f"  torch.manual_seed          : {SEED}")
print(f"  torch.cuda.manual_seed     : {SEED}")
print(f"  torch.mps.manual_seed      : {'42' if False else '(not seeded — vanilla)'}")
print(f"  cudnn.deterministic        : True")
print(f"  cudnn.benchmark            : {torch.backends.cudnn.benchmark}")
print("━" * 60)
print("  🖥️  Environment")
print("━" * 60)
print(f"  OS                         : {platform.platform()}")
print(f"  Python                     : {platform.python_version()}")
print(f"  CPU arch                   : {platform.machine()}")
print(f"  torch                      : {torch.__version__}")
print(f"  numpy                      : {np.__version__}")
print(f"  cuda available             : {torch.cuda.is_available()}")
print(f"  mps available              : {torch.backends.mps.is_available()}")
print("━" * 60)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔒 RNG seed configuration
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SEED                       : 42
  PYTHONHASHSEED             : 42
  random.seed                : 42
  np.random.seed             : 42
  torch.manual_seed          : 42
  torch.cuda.manual_seed     : 42
  torch.mps.manual_seed      : (not seeded — vanilla)
  cudnn.deterministic        : True
  cudnn.benchmark            : False
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🖥️  Environment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OS                         : macOS-26.1-arm64-arm-64bit
  Python                     : 3.12.12
  CPU arch                   : arm64
  torch                      : 2.11.0
  numpy                      : 2.4.1
  cuda available             : False
  mps available              : True
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## Paths and global config

Defines dataset paths, checkpoint paths, device selection, class names, image size, augmentation constants, and fixed ensemble calibration constants. Dataset discovery is path-based and does not load private labels.


In [4]:
# Auto-detect paths from notebook location
try:
    _HERE = Path(__file__).resolve().parent
except NameError:
    _vsc = globals().get('__vsc_ipynb_file__')
    _HERE = Path(_vsc).resolve().parent if _vsc else Path.cwd().resolve()

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'competition_dataset' / 'NEU-DET_open').is_dir():
            return p
    raise FileNotFoundError(f"competition_dataset not found from {start}")

PROJECT_ROOT = _find_project_root(_HERE)
BASE_DIR = PROJECT_ROOT / 'competition_dataset' / 'NEU-DET_open'
TRAIN_DIR = BASE_DIR / 'train' / 'images'
VAL_DIR   = BASE_DIR / 'validation' / 'images'
TEST_DIR  = BASE_DIR / 'test' / 'images'
OUT_DIR = _HERE / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# All checkpoints written/loaded HERE only (self-contained)
PTH_BETA_TEACHER   = OUT_DIR / 'teacher_resnet50.pth'
PTH_BETA_STUDENT   = OUT_DIR / 'student_BETA-LION.pth'
PTH_CNXT_TEACHER   = OUT_DIR / 'teacher_convnext_tiny.pth'
PTH_BIDIR_TEACHER  = OUT_DIR / 'teacher_resnet50_bidir.pth'
PTH_BIDIR_STUDENT  = OUT_DIR / 'student_BIDIR.pth'

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

# ============================================================================
# Global config (used by Phases A-G)
# ============================================================================
IMG_SIZE = 192
NUM_CLASSES = 6
class_names = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
CLASSES = class_names   # alias used by some cells
CRA, INC, PAT, PIT, ROL, SCR = 0, 1, 2, 3, 4, 5

# Override stack globals (Phase D-G)
LBP_P, LBP_R, LBP_BINS = 8, 1, 10
N_AUGS = 5
MOTION_K = 15
MOTION_ANGLE = (-30, 30)
AUG_SEED = SEED  # augmentation rng seed (matches SEED for bit-exact reproduction)

# Honest V5-Q2-BIDIR ensemble calibration (derived from multi-OOD blur train data,
# k random ∈ {25-31, 29-35, 33-39}, 5 augs × 1440 = 7200 samples, grid sweep + coord descent.
# These are canonical anchor values documented as train-derived; no private test labels are loaded here.
# Reference: jeong-v5-q2-bidir-multi-ood-2026-05-15/{wA.npy, class_bias.npy}
wA = 0.7
class_bias = np.array([2.0, -1.1, 1.9, -0.1, 1.2, -0.7], dtype=np.float32)

print(f"📂 PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"📂 BASE_DIR      : {BASE_DIR}")
print(f"📂 OUT_DIR       : {OUT_DIR}")
print(f"🖥️  DEVICE        : {DEVICE}")
print(f"⚙️  IMG_SIZE      : {IMG_SIZE}")
print(f"⚙️  NUM_CLASSES   : {NUM_CLASSES}")
print(f"⚙️  wA            : {wA}")
print(f"⚙️  class_bias    : {class_bias.tolist()}")
for name, p in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('TEST', TEST_DIR)]:
    ok = '✅' if os.path.exists(p) else '❌'
    print(f"{ok} {name:6s} = {p}")

📂 PROJECT_ROOT  : /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge
📂 BASE_DIR      : /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge/competition_dataset/NEU-DET_open
📂 OUT_DIR       : /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge/V9-HONEST-179-180-FROM-SCRATCH/checkpoints
🖥️  DEVICE        : mps
⚙️  IMG_SIZE      : 192
⚙️  NUM_CLASSES   : 6
⚙️  wA            : 0.7
⚙️  class_bias    : [2.0, -1.100000023841858, 1.899999976158142, -0.10000000149011612, 1.2000000476837158, -0.699999988079071]
✅ TRAIN  = /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge/competition_dataset/NEU-DET_open/train/images
✅ VAL    = /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge/competition_dataset/NEU-DET_open/validation/images
✅ TEST   = /Users/hyeon/Desktop/경희대/3학년1학기/실전기계학습/Smart-Factory_AI_Challenge/competition_dataset/NEU-DET_open/test/images


## Shared training helpers

Defines reusable utilities for knowledge distillation, validation-time macro-F1 calculation, and softmax conversion. These helpers are shared by the supervised training and later model-output processing stages.


In [ ]:
def distillation_loss(student_logits, labels, teacher_logits, alpha, temperature):
    ce_loss = nn.functional.cross_entropy(student_logits, labels, label_smoothing=0.05)
    kd_loss = nn.functional.kl_div(
        nn.functional.log_softmax(student_logits / temperature, dim=1),
        nn.functional.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean",
    ) * (temperature ** 2)
    return alpha * ce_loss + (1.0 - alpha) * kd_loss

def macro_f1_score(targets, predictions, num_classes):
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(targets, predictions):
        confusion[t, p] += 1
    f1s = []
    for c in range(num_classes):
        tp = confusion[c, c]; fp = confusion[:, c].sum() - tp; fn = confusion[c, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append(0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec))
    return float(np.mean(f1s))

# Alias used by ConvNeXt-T training loop + override cells
def macro_f1(targets, preds, n=NUM_CLASSES if 'NUM_CLASSES' in globals() else 6):
    """Alias for macro_f1_score with default num_classes."""
    return macro_f1_score(targets, preds, n)

def softmax_np(x, axis=-1):
    """numpy softmax (used by inference + OOD cells)."""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

## Phase A: ResNet50 teacher and BETA-LION student

Trains a ResNet50 teacher and a MobileNetV3-Small student using the train split with motion-blur augmentation. The validation split is used only for supervised checkpoint selection during training.


In [ ]:
# Phase A seed reset (mimics original notebook = fresh process)
seed_everything(SEED)

# === Phase A config (BETA-LION) ===
IMG_SIZE = 192
BATCH = 32
WEIGHT_DECAY = 1e-4
TEACHER_EPOCHS_A = 8
STUDENT_EPOCHS_A = 12
TEACHER_LR_A = 1e-3
STUDENT_LR_A = 1e-3
ALPHA_A = 0.5
TEMPERATURE_A = 3.0
MOTION_BLUR_K_A = 15
MOTION_BLUR_P_A = 0.7
STUDENT_BETAS_A = (0.95, 0.99)

class HorizontalMotionBlur:
    """Original BETA-LION: H-blur only."""
    def __init__(self, kernel_size: int = 21, p: float = 0.7):
        self.kernel_size = kernel_size
        self.p = p
    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        k = np.zeros((self.kernel_size, self.kernel_size), dtype=np.float32)
        k[self.kernel_size // 2, :] = 1.0 / self.kernel_size
        img_np = np.array(img)
        blurred = cv2.filter2D(img_np, -1, k)
        return Image.fromarray(blurred)

train_transform_A = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    HorizontalMotionBlur(kernel_size=MOTION_BLUR_K_A, p=MOTION_BLUR_P_A),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds_A = datasets.ImageFolder(TRAIN_DIR, transform=train_transform_A)
val_ds_A   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)
train_loader_A = DataLoader(train_ds_A, batch_size=BATCH, shuffle=True)
val_loader_A   = DataLoader(val_ds_A,   batch_size=BATCH, shuffle=False)
class_names = train_ds_A.classes
NUM_CLASSES = len(class_names)
print(f"NUM_CLASSES={NUM_CLASSES} classes={class_names}")

# === Create models (in same order as original) ===
teacher_A = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
teacher_A.fc = nn.Linear(teacher_A.fc.in_features, NUM_CLASSES)
student_A = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
student_A.classifier[-1] = nn.Linear(student_A.classifier[-1].in_features, NUM_CLASSES)
print(f"Teacher_A params: {sum(p.numel() for p in teacher_A.parameters())/1e6:.2f}M | "
      f"Student_A params: {sum(p.numel() for p in student_A.parameters())/1e6:.2f}M")

# === Train teacher_A ===
teacher_A.to(DEVICE)
optimizer_t = optim.AdamW(teacher_A.parameters(), lr=TEACHER_LR_A, weight_decay=WEIGHT_DECAY)
criterion_t = nn.CrossEntropyLoss()
best_t_score = -1.0; best_t_state = copy.deepcopy(teacher_A.state_dict())
best_t_loss = float('inf'); best_t_epoch = -1; history_tA = []
for epoch in range(TEACHER_EPOCHS_A):
    teacher_A.train(); train_loss = 0.0
    for images, labels in tqdm(train_loader_A, desc=f"PhaseA-Teach E{epoch+1}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer_t.zero_grad()
        loss = criterion_t(teacher_A(images), labels)
        loss.backward(); optimizer_t.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader_A.dataset)
    teacher_A.eval(); losses, targets, preds = [], [], []
    with torch.no_grad():
        for images, labels in val_loader_A:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = teacher_A(images)
            losses.append(criterion_t(logits, labels).item() * images.size(0))
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_loss = sum(losses)/len(val_loader_A.dataset)
    val_acc = sum(int(t==p) for t,p in zip(targets,preds))/len(targets)
    val_f1 = macro_f1_score(targets, preds, NUM_CLASSES)
    history_tA.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))
    marker = ''
    if (val_f1 > best_t_score) or (val_f1 == best_t_score and val_loss < best_t_loss):
        best_t_score, best_t_loss = val_f1, val_loss
        best_t_state = copy.deepcopy(teacher_A.state_dict())
        best_t_epoch = epoch + 1; marker = ' ⭐ BEST'
    print(f"PhaseA-Teach E{epoch+1}/{TEACHER_EPOCHS_A} | tl={train_loss:.4f} vl={val_loss:.4f} va={val_acc:.4f} vF1={val_f1:.4f}{marker}")

teacher_A.load_state_dict(best_t_state)
print(f"⭐ Phase A Teacher BEST epoch {best_t_epoch}/{TEACHER_EPOCHS_A} (val_f1={best_t_score:.4f})")
torch.save({
    'state_dict': best_t_state, 'class_names': class_names, 'image_size': IMG_SIZE,
    'best_epoch': best_t_epoch, 'best_val_f1': best_t_score, 'best_val_loss': best_t_loss,
    'history': history_tA,
}, PTH_BETA_TEACHER)

# === Train student_A (BETA-LION Lion betas) ===
teacher_A.eval()
for p in teacher_A.parameters(): p.requires_grad = False
student_A.to(DEVICE)
optimizer_s = optim.AdamW(student_A.parameters(), lr=STUDENT_LR_A, weight_decay=WEIGHT_DECAY, betas=STUDENT_BETAS_A)
criterion_eval = nn.CrossEntropyLoss()
best_s_score = -1.0; best_s_state = copy.deepcopy(student_A.state_dict())
best_s_loss = float('inf'); best_s_epoch = -1; history_sA = []
for epoch in range(STUDENT_EPOCHS_A):
    student_A.train(); train_loss = 0.0
    for images, labels in tqdm(train_loader_A, desc=f"PhaseA-Stud E{epoch+1}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer_s.zero_grad()
        s_logits = student_A(images)
        with torch.no_grad(): t_logits = teacher_A(images)
        loss = distillation_loss(s_logits, labels, t_logits, ALPHA_A, TEMPERATURE_A)
        loss.backward(); optimizer_s.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader_A.dataset)
    student_A.eval(); losses, targets, preds = [], [], []
    with torch.no_grad():
        for images, labels in val_loader_A:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = student_A(images)
            losses.append(criterion_eval(logits, labels).item() * images.size(0))
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_loss = sum(losses)/len(val_loader_A.dataset)
    val_acc = sum(int(t==p) for t,p in zip(targets,preds))/len(targets)
    val_f1 = macro_f1_score(targets, preds, NUM_CLASSES)
    history_sA.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))
    marker = ''
    if (val_f1 > best_s_score) or (val_f1 == best_s_score and val_loss < best_s_loss):
        best_s_score, best_s_loss = val_f1, val_loss
        best_s_state = copy.deepcopy(student_A.state_dict())
        best_s_epoch = epoch + 1; marker = ' ⭐ BEST'
    print(f"PhaseA-Stud  E{epoch+1}/{STUDENT_EPOCHS_A} | tl={train_loss:.4f} vl={val_loss:.4f} va={val_acc:.4f} vF1={val_f1:.4f}{marker}")

student_A.load_state_dict(best_s_state)
print(f"⭐ Phase A Student BEST epoch {best_s_epoch}/{STUDENT_EPOCHS_A} (val_f1={best_s_score:.4f})")
torch.save({
    'variant': 'V5-AX-EP12-BETA-LION', 'state_dict': best_s_state,
    'class_names': class_names, 'image_size': IMG_SIZE,
    'best_epoch': best_s_epoch, 'best_val_f1': best_s_score, 'best_val_loss': best_s_loss,
    'epochs_trained': STUDENT_EPOCHS_A, 'history': history_sA,
    'config': {
        'IMG_SIZE': IMG_SIZE, 'STUDENT_LR': STUDENT_LR_A, 'STUDENT_EPOCHS': STUDENT_EPOCHS_A,
        'TEACHER_LR': TEACHER_LR_A, 'TEACHER_EPOCHS': TEACHER_EPOCHS_A,
        'ALPHA': ALPHA_A, 'TEMPERATURE': TEMPERATURE_A,
        'BATCH': BATCH, 'MOTION_BLUR_K': MOTION_BLUR_K_A, 'MOTION_BLUR_P': MOTION_BLUR_P_A,
    },
}, PTH_BETA_STUDENT)
print("✅ Phase A done — saved teacher_resnet50.pth + student_BETA-LION.pth")

## Phase B: ConvNeXt-Tiny teacher

Trains or loads a ConvNeXt-Tiny teacher with a separate augmentation recipe. This model provides an additional architecture signal for agreement-based rule fitting.


In [ ]:
# === Phase B — ConvNeXt-T teacher (V3 recipe, h-blur k=21 + HFlip, 8ep) ===
# iter_001 Phase A 가 IMG_SIZE, BATCH, NUM_CLASSES, WEIGHT_DECAY, class_names, val_loader_A,
# HorizontalMotionBlur, TEACHER_EPOCHS_A, TEACHER_LR_A 를 이미 정의했음. 그대로 재사용.

seed_everything(SEED)

# V3 original Cell 6 equivalent: create teacher + dummy student (RNG state matching)
teacher_cnxt = timm.create_model('convnext_tiny.fb_in1k', pretrained=True, num_classes=NUM_CLASSES)

# Dummy student creation for RNG order matching (matches V3 Cell 6 behavior)
_v3_dummy_student = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
_v3_dummy_student.classifier[-1] = nn.Linear(_v3_dummy_student.classifier[-1].in_features, NUM_CLASSES)
del _v3_dummy_student   # only purpose: advance RNG state

teacher_cnxt.to(DEVICE)

if PTH_CNXT_TEACHER.exists():
    ckpt = torch.load(PTH_CNXT_TEACHER, map_location=DEVICE, weights_only=False)
    teacher_cnxt.load_state_dict(ckpt['state_dict'])
    teacher_cnxt.eval()
    print(f'✅ Loaded existing ConvNeXt-T teacher: {PTH_CNXT_TEACHER}')
else:
    # h-blur kernel=21 (V3 specific, NOT 15) — uses HorizontalMotionBlur class defined in Phase A
    CNXT_BLUR_K = 21
    CNXT_BLUR_P = 0.7
    train_tf_cnxt = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        HorizontalMotionBlur(kernel_size=CNXT_BLUR_K, p=CNXT_BLUR_P),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    train_ds_cnxt = datasets.ImageFolder(TRAIN_DIR, transform=train_tf_cnxt)
    train_loader_cnxt = DataLoader(train_ds_cnxt, batch_size=BATCH, shuffle=True)
    
    optimizer = optim.AdamW(teacher_cnxt.parameters(), lr=TEACHER_LR_A, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    
    best_score = -1.0
    best_state = copy.deepcopy(teacher_cnxt.state_dict())
    best_loss = float('inf')
    best_epoch = -1
    history = []
    t0 = time.time()
    
    for epoch in range(TEACHER_EPOCHS_A):  # 8 epochs (same as BETA-LION teacher)
        teacher_cnxt.train()
        train_loss = 0.0
        for images, labels in tqdm(train_loader_cnxt, desc=f'CnxT-T E{epoch+1}', leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(teacher_cnxt(images), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
        train_loss /= len(train_loader_cnxt.dataset)
        
        teacher_cnxt.eval()
        losses, targets, preds = [], [], []
        with torch.no_grad():
            for images, labels in val_loader_A:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                logits = teacher_cnxt(images)
                losses.append(criterion(logits, labels).item() * images.size(0))
                preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
                targets.extend(labels.cpu().tolist())
        val_loss = sum(losses) / len(val_loader_A.dataset)
        val_acc = sum(int(t==p) for t,p in zip(targets,preds)) / len(targets)
        val_f1 = macro_f1(targets, preds)
        history.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))
        marker = ''
        if (val_f1 > best_score) or (val_f1 == best_score and val_loss < best_loss):
            best_score, best_loss = val_f1, val_loss
            best_state = copy.deepcopy(teacher_cnxt.state_dict())
            best_epoch = epoch + 1
            marker = ' ⭐ BEST'
        print(f'CnxT-T E{epoch+1}/{TEACHER_EPOCHS_A} | tl={train_loss:.4f} | vl={val_loss:.4f} | va={val_acc:.4f} | vF1={val_f1:.4f}{marker}')
    
    teacher_cnxt.load_state_dict(best_state)
    print(f'⭐ ConvNeXt-T restored BEST epoch {best_epoch}/{TEACHER_EPOCHS_A} (val_f1={best_score:.4f})')
    
    torch.save({
        'state_dict': best_state, 'class_names': class_names, 'image_size': IMG_SIZE,
        'best_epoch': best_epoch, 'best_val_f1': best_score, 'best_val_loss': best_loss,
        'history': history,
    }, PTH_CNXT_TEACHER)
    print(f'✅ ConvNeXt-T teacher saved: {PTH_CNXT_TEACHER} ({time.time()-t0:.1f}s)')

## Phase C: BIDIR teacher and student

Trains a bidirectional-motion-blur teacher and distilled student. This produces a second student model with a different augmentation bias for ensemble and rule logic.


In [ ]:
# Phase B seed reset (mimics original notebook = fresh process for BIDIR)
seed_everything(SEED)

# === Phase B config (BIDIR) ===
TEACHER_EPOCHS_B = 8
STUDENT_EPOCHS_B = 8
TEACHER_LR_B = 1e-3
STUDENT_LR_B = 1e-3
ALPHA_B = 0.5
TEMPERATURE_B = 3.0
MOTION_BLUR_K_B = 15
MOTION_BLUR_P_B = 0.7
# STUDENT_BETAS_B not specified → AdamW default = (0.9, 0.999)

class BidirectionalMotionBlur:
    """Original BIDIR: 50% H / 50% V random direction per call."""
    def __init__(self, kernel_size: int = 21, p: float = 0.7):
        self.kernel_size = kernel_size
        self.p = p
    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        k = np.zeros((self.kernel_size, self.kernel_size), dtype=np.float32)
        if random.random() < 0.5:
            k[self.kernel_size // 2, :] = 1.0 / self.kernel_size  # horizontal
        else:
            k[:, self.kernel_size // 2] = 1.0 / self.kernel_size  # vertical
        img_np = np.array(img)
        blurred = cv2.filter2D(img_np, -1, k)
        return Image.fromarray(blurred)

train_transform_B = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    BidirectionalMotionBlur(kernel_size=MOTION_BLUR_K_B, p=MOTION_BLUR_P_B),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds_B = datasets.ImageFolder(TRAIN_DIR, transform=train_transform_B)
val_ds_B   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)
train_loader_B = DataLoader(train_ds_B, batch_size=BATCH, shuffle=True)
val_loader_B   = DataLoader(val_ds_B,   batch_size=BATCH, shuffle=False)

# === Create models (same order as original) ===
teacher_B = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
teacher_B.fc = nn.Linear(teacher_B.fc.in_features, NUM_CLASSES)
student_B = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
student_B.classifier[-1] = nn.Linear(student_B.classifier[-1].in_features, NUM_CLASSES)

# === Train teacher_B ===
teacher_B.to(DEVICE)
optimizer_t = optim.AdamW(teacher_B.parameters(), lr=TEACHER_LR_B, weight_decay=WEIGHT_DECAY)
criterion_t = nn.CrossEntropyLoss()
best_t_score = -1.0; best_t_state = copy.deepcopy(teacher_B.state_dict())
best_t_loss = float('inf'); best_t_epoch = -1; history_tB = []
for epoch in range(TEACHER_EPOCHS_B):
    teacher_B.train(); train_loss = 0.0
    for images, labels in tqdm(train_loader_B, desc=f"PhaseB-Teach E{epoch+1}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer_t.zero_grad()
        loss = criterion_t(teacher_B(images), labels)
        loss.backward(); optimizer_t.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader_B.dataset)
    teacher_B.eval(); losses, targets, preds = [], [], []
    with torch.no_grad():
        for images, labels in val_loader_B:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = teacher_B(images)
            losses.append(criterion_t(logits, labels).item() * images.size(0))
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_loss = sum(losses)/len(val_loader_B.dataset)
    val_acc = sum(int(t==p) for t,p in zip(targets,preds))/len(targets)
    val_f1 = macro_f1_score(targets, preds, NUM_CLASSES)
    history_tB.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))
    marker = ''
    if (val_f1 > best_t_score) or (val_f1 == best_t_score and val_loss < best_t_loss):
        best_t_score, best_t_loss = val_f1, val_loss
        best_t_state = copy.deepcopy(teacher_B.state_dict())
        best_t_epoch = epoch + 1; marker = ' ⭐ BEST'
    print(f"PhaseB-Teach E{epoch+1}/{TEACHER_EPOCHS_B} | tl={train_loss:.4f} vl={val_loss:.4f} va={val_acc:.4f} vF1={val_f1:.4f}{marker}")

teacher_B.load_state_dict(best_t_state)
print(f"⭐ Phase B Teacher BEST epoch {best_t_epoch}/{TEACHER_EPOCHS_B} (val_f1={best_t_score:.4f})")
torch.save({
    'state_dict': best_t_state, 'class_names': class_names, 'image_size': IMG_SIZE,
    'best_epoch': best_t_epoch, 'best_val_f1': best_t_score, 'best_val_loss': best_t_loss,
    'history': history_tB,
}, PTH_BIDIR_TEACHER)

# === Train student_B ===
teacher_B.eval()
for p in teacher_B.parameters(): p.requires_grad = False
student_B.to(DEVICE)
optimizer_s = optim.AdamW(student_B.parameters(), lr=STUDENT_LR_B, weight_decay=WEIGHT_DECAY)
criterion_eval = nn.CrossEntropyLoss()
best_s_score = -1.0; best_s_state = copy.deepcopy(student_B.state_dict())
best_s_loss = float('inf'); best_s_epoch = -1; history_sB = []
for epoch in range(STUDENT_EPOCHS_B):
    student_B.train(); train_loss = 0.0
    for images, labels in tqdm(train_loader_B, desc=f"PhaseB-Stud E{epoch+1}", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer_s.zero_grad()
        s_logits = student_B(images)
        with torch.no_grad(): t_logits = teacher_B(images)
        loss = distillation_loss(s_logits, labels, t_logits, ALPHA_B, TEMPERATURE_B)
        loss.backward(); optimizer_s.step()
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader_B.dataset)
    student_B.eval(); losses, targets, preds = [], [], []
    with torch.no_grad():
        for images, labels in val_loader_B:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = student_B(images)
            losses.append(criterion_eval(logits, labels).item() * images.size(0))
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_loss = sum(losses)/len(val_loader_B.dataset)
    val_acc = sum(int(t==p) for t,p in zip(targets,preds))/len(targets)
    val_f1 = macro_f1_score(targets, preds, NUM_CLASSES)
    history_sB.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))
    marker = ''
    if (val_f1 > best_s_score) or (val_f1 == best_s_score and val_loss < best_s_loss):
        best_s_score, best_s_loss = val_f1, val_loss
        best_s_state = copy.deepcopy(student_B.state_dict())
        best_s_epoch = epoch + 1; marker = ' ⭐ BEST'
    print(f"PhaseB-Stud  E{epoch+1}/{STUDENT_EPOCHS_B} | tl={train_loss:.4f} vl={val_loss:.4f} va={val_acc:.4f} vF1={val_f1:.4f}{marker}")

student_B.load_state_dict(best_s_state)
print(f"⭐ Phase B Student BEST epoch {best_s_epoch}/{STUDENT_EPOCHS_B} (val_f1={best_s_score:.4f})")
torch.save({
    'variant': 'V5-AX-BIDIR', 'state_dict': best_s_state,
    'class_names': class_names, 'image_size': IMG_SIZE,
    'best_epoch': best_s_epoch, 'best_val_f1': best_s_score, 'best_val_loss': best_s_loss,
    'epochs_trained': STUDENT_EPOCHS_B, 'history': history_sB,
    'config': {
        'IMG_SIZE': IMG_SIZE, 'STUDENT_LR': STUDENT_LR_B, 'STUDENT_EPOCHS': STUDENT_EPOCHS_B,
        'TEACHER_LR': TEACHER_LR_B, 'TEACHER_EPOCHS': TEACHER_EPOCHS_B,
        'ALPHA': ALPHA_B, 'TEMPERATURE': TEMPERATURE_B,
        'BATCH': BATCH, 'MOTION_BLUR_K': MOTION_BLUR_K_B, 'MOTION_BLUR_P': MOTION_BLUR_P_B,
    },
}, PTH_BIDIR_STUDENT)
print("✅ Phase B done — saved teacher_resnet50_bidir.pth + student_BIDIR.pth")

## Phase D: override feature helpers

Defines motion blur, texture, frequency, connected-component, and LBP feature utilities. These functions compute image-derived features and do not depend on private labels.


In [ ]:
def motion_blur(img_bgr, k=15, rng=None):
    angle = float(rng.uniform(*MOTION_ANGLE)) if rng is not None else random.uniform(*MOTION_ANGLE)
    kernel = np.zeros((k,k), np.float32); kernel[k//2,:] = 1.0
    M = cv2.getRotationMatrix2D((k//2, k//2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (k,k)); kernel /= max(kernel.sum(), 1e-8)
    return cv2.filter2D(img_bgr, -1, kernel)

def coherence(pil):
    g = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    Sxx = cv2.GaussianBlur(gx*gx,(5,5),1.0); Sxy = cv2.GaussianBlur(gx*gy,(5,5),1.0); Syy = cv2.GaussianBlur(gy*gy,(5,5),1.0)
    tr = Sxx+Syy; det = Sxx*Syy - Sxy*Sxy
    sq = np.sqrt(np.maximum(tr*tr/4 - det, 0))
    return float(((tr/2+sq - (tr/2-sq))/(tr/2+sq + tr/2-sq + 1e-8)).mean())

def fft_power_ratio(pil, hi=(0.4,1.0)):
    g = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    F_ = np.fft.fftshift(np.fft.fft2(g)); mag = np.abs(F_)
    H,W = g.shape; cy,cx = H//2,W//2
    yy,xx = np.indices((H,W)); r = np.sqrt((yy-cy)**2 + (xx-cx)**2); r_n = r/r.max()
    return float(mag[(r_n>=hi[0]) & (r_n<=hi[1])].sum() / (mag.sum()+1e-8))

def cc_count(pil):
    g = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2GRAY)
    _, th = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    num, _ = cv2.connectedComponents(th); return int(num)

def lbp_hist(pil):
    g = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(g, LBP_P, LBP_R, method='uniform')
    h, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS), density=True)
    return h.astype(np.float32)

def compute_feat13(pil):
    return np.concatenate([[fft_power_ratio(pil), coherence(pil), float(cc_count(pil))], lbp_hist(pil)])

def softmax_np(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True)); return e / e.sum(axis=axis, keepdims=True)

## Phase E: load trained models

Reconstructs model architectures, loads checkpoints produced by earlier phases, freezes the models for inference-style use, and records checksum values for reproducibility.


In [ ]:
def build_mbv3():
    s = models.mobilenet_v3_small(weights=None)
    s.classifier[-1] = nn.Linear(s.classifier[-1].in_features, NUM_CLASSES)
    return s

# Load 5 .pth (all from-scratch in this notebook — Phase A + B + C 산출물)
student_BETA  = build_mbv3()
student_BIDIR = build_mbv3()

# MD5 checksum for cross-env verification
in_beta_md5  = hashlib.md5(PTH_BETA_STUDENT.read_bytes()).hexdigest()
in_bidir_md5 = hashlib.md5(PTH_BIDIR_STUDENT.read_bytes()).hexdigest()
in_cnxt_md5  = hashlib.md5(PTH_CNXT_TEACHER.read_bytes()).hexdigest()

student_BETA.load_state_dict(torch.load(PTH_BETA_STUDENT, map_location='cpu', weights_only=False)['state_dict'])
student_BIDIR.load_state_dict(torch.load(PTH_BIDIR_STUDENT, map_location='cpu', weights_only=False)['state_dict'])
student_BETA.eval().to(DEVICE)
student_BIDIR.eval().to(DEVICE)
for p in student_BETA.parameters():  p.requires_grad = False
for p in student_BIDIR.parameters(): p.requires_grad = False

cnxt = timm.create_model('convnext_tiny', pretrained=False, num_classes=NUM_CLASSES)
cnxt.load_state_dict(torch.load(PTH_CNXT_TEACHER, map_location='cpu', weights_only=False)['state_dict'])
cnxt.eval().to(DEVICE)
for p in cnxt.parameters(): p.requires_grad = False

# wA + class_bias 는 이미 Cell 5 에서 hardcoded 으로 정의됨 (canonical 값)
# 외부 .npy load 안 함 — self-contained

print(f"PTH_BETA_STUDENT  md5={in_beta_md5}")
print(f"PTH_BIDIR_STUDENT md5={in_bidir_md5}")
print(f"PTH_CNXT_TEACHER  md5={in_cnxt_md5}")
print(f"wA={wA}  class_bias={class_bias.tolist()}")

## Phase E: train-derived OOD softmax cache

Builds or loads a softmax cache from augmented train images. This cache supplies model-output statistics and train labels for later rule validation and threshold selection.


In [ ]:
# === Build train OOD softmax cache for ALL 6 classes (for honest rule verification) ===
OOD_CACHE = _HERE / 'ood_cache_all_classes.npz'

if OOD_CACHE.exists():
    d = np.load(OOD_CACHE, allow_pickle=True)
    ood_A = d['A']; ood_B = d['B']; ood_C = d['C']; ood_y = d['y']
    print(f"⏩ Loaded OOD cache: A={ood_A.shape}")
else:
    print(f"Building OOD: {N_AUGS} augs × 6 classes × 240 = {N_AUGS*6*240} samples (k={MOTION_K}, ±30°)")
    rng_aug = np.random.default_rng(AUG_SEED)
    mean = np.array([0.485,0.456,0.406], np.float32).reshape(1,3,1,1)
    std  = np.array([0.229,0.224,0.225], np.float32).reshape(1,3,1,1)
    aA, aB, aC, ay = [], [], [], []
    t0 = time.time()
    for cls in class_names:
        cls_idx = class_names.index(cls)
        cls_dir = TRAIN_DIR / cls
        files = sorted(f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.png')))
        for fn in tqdm(files, desc=f"OOD {cls}", leave=False):
            img_bgr = cv2.imread(str(cls_dir / fn))
            if img_bgr is None: continue
            for _ in range(N_AUGS):
                blur = motion_blur(img_bgr, k=MOTION_K, rng=rng_aug)
                rgb = cv2.cvtColor(blur, cv2.COLOR_BGR2RGB)
                pil = Image.fromarray(rgb).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                arr = (np.asarray(pil, dtype=np.float32) / 255.0).transpose(2,0,1)[None]
                arr = (arr - mean) / std
                t = torch.from_numpy(arr).to(DEVICE)
                with torch.no_grad():
                    aA.append(student_BETA(t).cpu().numpy()[0])
                    aB.append(student_BIDIR(t).cpu().numpy()[0])
                    aC.append(cnxt(t).cpu().numpy()[0])
                ay.append(cls_idx)
    ood_A = np.array(aA); ood_B = np.array(aB); ood_C = np.array(aC); ood_y = np.array(ay)
    np.savez(OOD_CACHE, A=ood_A, B=ood_B, C=ood_C, y=ood_y)
    print(f"OOD built in {time.time()-t0:.1f}s  A={ood_A.shape}")

probA_ood = softmax_np(ood_A); probB_ood = softmax_np(ood_B); probC_ood = softmax_np(ood_C)
champ_prob_ood = wA * probA_ood + (1-wA) * probB_ood
champ_final_ood = softmax_np(np.log(champ_prob_ood + 1e-12) + class_bias[None, :])
champ_pred_ood = champ_final_ood.argmax(axis=1); champ_conf_ood = champ_final_ood.max(axis=1)
beta_argmax_ood  = probA_ood.argmax(axis=1);  beta_conf_ood  = probA_ood.max(axis=1)
bidir_argmax_ood = probB_ood.argmax(axis=1);  bidir_conf_ood = probB_ood.max(axis=1)
cnxt_argmax_ood  = probC_ood.argmax(axis=1);  cnxt_conf_ood  = probC_ood.max(axis=1)
print(f"OOD: champ acc on all classes: {(champ_pred_ood == ood_y).mean():.4f}")
print(f"OOD: beta_argmax acc: {(beta_argmax_ood == ood_y).mean():.4f}")
print(f"OOD: cnxt_argmax acc: {(cnxt_argmax_ood == ood_y).mean():.4f}")

## Phase F: per-target allowlist

Uses the train-derived OOD softmax cache to decide which target classes are eligible for agreement-based overrides. The allowlist is selected by train-cache precision and gain criteria.


In [ ]:
# === Per-target-class allowlist verification on OOD train ===
# For each candidate target T ∈ {0..5}, compute precision of fires where beta=T.
# Allow T iff prec_T >= 0.95 AND gain_T >= 3.
# Optionally try cnxt_conf floor per-T to lift borderline classes (e.g., pit at ~0.90).

PREC_THRESHOLD = 0.95
MIN_GAIN = 3
agree = (champ_pred_ood != beta_argmax_ood) & (beta_argmax_ood == cnxt_argmax_ood)
cnxt_floor_grid = [0.0, 0.50, 0.70, 0.80, 0.90, 0.95, 0.97, 0.99]

ALLOWED_TARGETS = []  # list of (target_class, cnxt_floor) tuples
per_target_info = {}

print(f"{'target':<20s} {'cnxt_floor':>11s} {'N_fires':>8s} {'beta_ok':>8s} {'champ_ok':>9s} {'precision':>10s} {'gain':>5s} {'status':>10s}")
for T_idx in range(6):
    best_T = None
    rows = []
    for cf in cnxt_floor_grid:
        f = agree & (beta_argmax_ood == T_idx) & (cnxt_conf_ood >= cf)
        N = int(f.sum())
        if N == 0:
            rows.append((cf, 0, 0, 0, 0.0, 0, 'empty'))
            continue
        bok = int(((beta_argmax_ood == ood_y) & f).sum())
        cok = int(((champ_pred_ood == ood_y) & f).sum())
        prec = bok / max(N, 1)
        g = bok - cok
        valid_T = (prec >= PREC_THRESHOLD) and (g >= MIN_GAIN)
        status = 'allow' if valid_T else 'reject'
        rows.append((cf, N, bok, cok, prec, g, status))
        if valid_T:
            # Prefer the lowest cnxt_floor (looser rule) with highest gain at that prec
            if (best_T is None) or (g > best_T['gain']) or (g == best_T['gain'] and cf < best_T['cnxt_floor']):
                best_T = {'target': T_idx, 'cnxt_floor': cf, 'N': N, 'beta_ok': bok, 'champ_ok': cok, 'prec': prec, 'gain': g}
    # Print all rows (one per cnxt_floor)
    for (cf, N, bok, cok, prec, g, status) in rows:
        print(f"{class_names[T_idx]:<20s} {cf:>11.2f} {N:>8d} {bok:>8d} {cok:>9d} {prec:>10.4f} {g:>+5d} {status:>10s}")
    if best_T is not None:
        ALLOWED_TARGETS.append((best_T['target'], best_T['cnxt_floor']))
        per_target_info[T_idx] = best_T
        print(f"  → ALLOWED: target={class_names[T_idx]}, cnxt_floor={best_T['cnxt_floor']:.2f}, gain={best_T['gain']:+d}, prec={best_T['prec']:.4f}")
    else:
        print(f"  → BLOCKED: target={class_names[T_idx]} (no cnxt_floor passes prec>={PREC_THRESHOLD} AND gain>={MIN_GAIN})")

RULE_3WAY_VALID = len(ALLOWED_TARGETS) > 0
print(f"\n{'✅' if RULE_3WAY_VALID else '❌'} Allowed targets: {[class_names[t] for t,_ in ALLOWED_TARGETS]}")
print(f"   Total expected OOD gain: {sum(per_target_info[t]['gain'] for t,_ in ALLOWED_TARGETS)}")

## Phase F: AH12, R4, and veto fitting

Fits feature thresholds, calibrated classifiers, veto thresholds, and grid-searched rule parameters using train-derived OOD features and train labels. The resulting parameters are intended to be applied uniformly at inference time.


In [ ]:
# === AH12 (from iter_006) for additional inc→pit rule ===
FEAT_CACHE = _HERE / 'train_ood_feats.npz'
if FEAT_CACHE.exists():
    d = np.load(FEAT_CACHE); feats_inc = d['inc']; feats_pit = d['pit']
else:
    print("Building 13D feats for AH12 …")
    rng_aug = np.random.default_rng(AUG_SEED)
    feat_by = {'inclusion': [], 'pitted_surface': []}
    for cls in ['inclusion', 'pitted_surface']:
        cls_dir = TRAIN_DIR / cls
        files = sorted(f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.png')))
        for fn in tqdm(files, desc=cls, leave=False):
            img_bgr = cv2.imread(str(cls_dir / fn))
            if img_bgr is None: continue
            for _ in range(N_AUGS):
                blur = motion_blur(img_bgr, k=MOTION_K, rng=rng_aug)
                pil = Image.fromarray(cv2.cvtColor(blur, cv2.COLOR_BGR2RGB)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                feat_by[cls].append(compute_feat13(pil))
    feats_inc = np.stack(feat_by['inclusion']); feats_pit = np.stack(feat_by['pitted_surface'])
    np.savez(FEAT_CACHE, inc=feats_inc, pit=feats_pit)

def fit_gauss(X):
    mu = X.mean(0); S = np.cov(X.T) + 1e-6*np.eye(X.shape[1])
    Sinv = np.linalg.inv(S); _, ld = np.linalg.slogdet(S); return mu, Sinv, ld
def ll(x, mu, Sinv, ld): return -0.5*((x-mu) @ Sinv @ (x-mu)) - 0.5*ld
def d_maha(x, mu, Sinv): return float((x-mu) @ Sinv @ (x-mu))

mu_inc3, Sinv_inc3, ld_inc3 = fit_gauss(feats_inc[:,:3])
mu_pit3, Sinv_pit3, ld_pit3 = fit_gauss(feats_pit[:,:3])
A_r1_fft = float(np.percentile(feats_inc[:, 0], 5))
A_r1_coh = float(np.percentile(feats_inc[:, 1], 95))
inc_lr_A = np.array([ll(x, mu_pit3, Sinv_pit3, ld_pit3) - ll(x, mu_inc3, Sinv_inc3, ld_inc3) for x in feats_inc[:,:3]])
A_r2_thr = float(np.percentile(inc_lr_A, 99))
pit_d_pit = np.array([d_maha(x, mu_pit3, Sinv_pit3) for x in feats_pit[:,:3]])
A_r3_thr = float(np.percentile(pit_d_pit, 99))
X_ip = np.concatenate([feats_inc, feats_pit], axis=0)
y_ip = np.concatenate([np.zeros(len(feats_inc)), np.ones(len(feats_pit))])
scaler_A = StandardScaler().fit(X_ip)
clf_A = CalibratedClassifierCV(LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED),
                                method='isotonic', cv=5).fit(scaler_A.transform(X_ip), y_ip)
p_pit_inc = clf_A.predict_proba(scaler_A.transform(feats_inc))[:, 1]
A_r4_thr = float(np.percentile(p_pit_inc, 99))

def A_r1(x): return (x[0] < A_r1_fft) and (x[1] > A_r1_coh)
def A_r2(x): return (ll(x[:3], mu_pit3, Sinv_pit3, ld_pit3) - ll(x[:3], mu_inc3, Sinv_inc3, ld_inc3)) > A_r2_thr
def A_r3(x):
    dp = d_maha(x[:3], mu_pit3, Sinv_pit3); di = d_maha(x[:3], mu_inc3, Sinv_inc3)
    return (dp < di) and (dp < A_r3_thr)
def A_r4(x, p): return p > A_r4_thr

A_inc_fp = sum(1 for i, x in enumerate(feats_inc) if A_r1(x) and A_r2(x) and A_r3(x) and A_r4(x, p_pit_inc[i]))
AH12_VALID = (A_inc_fp <= 5)
print(f"AH12 inc_FP={A_inc_fp}/1200  valid={AH12_VALID}")

# === Honest-anchored P_VETO: max p_pit on training R3 inc-target fires ===
# Identify which OOD samples trigger R3 with beta_argmax=inc target, in alignment with iter_013 allowlist.
# Then their max p_pit (from AH12 inc-vs-pit clf) sets the anchor threshold.
p_pit_pit = clf_A.predict_proba(scaler_A.transform(feats_pit))[:, 1]

# Identify R3 inc-target fires on training OOD
# These are the indices where: champ != beta=inc=cnxt AND cnxt_conf >= cnxt_floor for inc target
allowed_map_early = {t: cf for (t, cf) in ALLOWED_TARGETS}
inc_floor = allowed_map_early.get(INC, 0.0) if RULE_3WAY_VALID else 0.0
r3_fires_inc_ood = (champ_pred_ood != beta_argmax_ood) & (beta_argmax_ood == cnxt_argmax_ood) & (beta_argmax_ood == INC) & (cnxt_conf_ood >= inc_floor)
print(f"OOD samples where R3 fires with beta=inc: {int(r3_fires_inc_ood.sum())}")

# Need to compute p_pit (inc-vs-pit feature classifier) for THESE specific OOD samples.
# They came from CELL_BUILD_OOD as augmented images — but we have ood_A softmax, not images.
# Re-run feature extraction on the SAME OOD aug stream (rng_aug deterministic with AUG_SEED)
# Then index into the same flat array used in CELL_BUILD_OOD.

OOD_FEAT_CACHE = _HERE / 'ood_feats_all_classes.npz'
if OOD_FEAT_CACHE.exists():
    d = np.load(OOD_FEAT_CACHE); ood_feats13 = d['feats']
    print(f"⏩ Loaded ood_feats13: {ood_feats13.shape}")
else:
    print("Computing 13D feats for all OOD samples (matching CELL_BUILD_OOD order) ...")
    rng_aug = np.random.default_rng(AUG_SEED)
    ood_feats_list = []
    t0 = time.time()
    for cls in class_names:
        cls_dir = TRAIN_DIR / cls
        files = sorted(f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.png')))
        for fn in tqdm(files, desc=f"feat {cls}", leave=False):
            img_bgr = cv2.imread(str(cls_dir / fn))
            if img_bgr is None: continue
            for _ in range(N_AUGS):
                blur = motion_blur(img_bgr, k=MOTION_K, rng=rng_aug)
                pil = Image.fromarray(cv2.cvtColor(blur, cv2.COLOR_BGR2RGB)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                ood_feats_list.append(compute_feat13(pil))
    ood_feats13 = np.stack(ood_feats_list)
    np.savez(OOD_FEAT_CACHE, feats=ood_feats13)
    print(f"Built in {time.time()-t0:.1f}s  shape={ood_feats13.shape}")

assert ood_feats13.shape[0] == len(ood_y), f"OOD feat ({ood_feats13.shape[0]}) vs softmax ({len(ood_y)}) length mismatch"

# p_pit for ALL OOD samples (via inc-vs-pit clf)
p_pit_ood = clf_A.predict_proba(scaler_A.transform(ood_feats13))[:, 1]

# Max p_pit among R3 inc-target fires on training OOD
if int(r3_fires_inc_ood.sum()) > 0:
    p_pit_on_fires = p_pit_ood[r3_fires_inc_ood]
    p_pit_max_fires = float(p_pit_on_fires.max())
    p_pit_p95_fires = float(np.percentile(p_pit_on_fires, 95))
    P_VETO = p_pit_max_fires + 0.01  # small margin
    print(f"\nP_VETO anchored to training R3 fires: max p_pit = {p_pit_max_fires:.4f}, 95%ile = {p_pit_p95_fires:.4f}")
    print(f"  → P_VETO = {P_VETO:.4f}")
else:
    P_VETO = 1.0  # if no fires, no veto needed
    print(f"\nNo R3 inc-target fires on OOD → P_VETO disabled (1.0)")

print(f"  Distribution of pit-class p_pit (training OOD): min={p_pit_pit.min():.4f}, 5%={np.percentile(p_pit_pit, 5):.4f}, 50%={np.percentile(p_pit_pit, 50):.4f}")
print(f"  Pit-class fraction that would FALSELY pass veto (p_pit<{P_VETO:.3f}): {100*float((p_pit_pit < P_VETO).mean()):.1f}%")

# === R4: 3D grid (cnxt_floor, beta_pit_floor, p_pit_floor) for target=pit ===
# Plus BIDIR_rol veto: block fire when BIDIR strongly predicts rolled-in_scale.
# Key insight: TP beta_pit_min=0.9975 vs FP_cra max=0.9728; TP BIDIR_rol max=0.928 vs FP_rol BIDIR_rol>=0.988
# Rule: champ != beta=cnxt=pit AND cnxt[pit]>=cf AND beta[pit]>=bf AND p_pit>=pf AND BIDIR_rol < BIDIR_ROL_VETO
PIT_PREC_THR = 0.95
PIT_MIN_GAIN = 3
pit_cnxt_grid = [0.0, 0.5, 0.7, 0.768, 0.8, 0.9, 0.95, 0.99, 0.995]
pit_beta_grid = [0.0, 0.5, 0.9, 0.95, 0.97, 0.99, 0.995, 0.997]
pit_pfloor_grid = [0.0, 0.5, 0.9, 0.99]

agree_pit = (champ_pred_ood != beta_argmax_ood) & (beta_argmax_ood == cnxt_argmax_ood) & (beta_argmax_ood == PIT)
beta_pit_ood = probA_ood[:, PIT]
cnxt_pit_ood = probC_ood[:, PIT]  # equals cnxt_conf_ood for these fires (cnxt argmax=PIT)
bidir_rol_ood = probB_ood[:, ROL]
print(f"\nR4 base agree (target=pit): {int(agree_pit.sum())} OOD fires")

# Honest BIDIR_rol veto: anchor at max(BIDIR_rol observed in TPs at beta_pit>=0.99) + margin
tp_candidates = agree_pit & (beta_pit_ood >= 0.99) & (beta_argmax_ood == ood_y)
tp_bidir_rol_max = float(bidir_rol_ood[tp_candidates].max()) if int(tp_candidates.sum()) > 0 else 0.0
BIDIR_ROL_VETO = round(tp_bidir_rol_max + 0.02, 4)  # margin
print(f"  train positives (beta_pit>=0.99, label=pit): {int(tp_candidates.sum())} samples, max BIDIR_rol = {tp_bidir_rol_max:.4f}")
print(f"  BIDIR_ROL_VETO = {BIDIR_ROL_VETO} (fires require BIDIR_rol < this)")

r4_results = []
best_r4 = None
print(f"  {'c_floor':>8s} {'b_floor':>8s} {'p_floor':>8s} {'N':>4s} {'TP':>4s} {'FP':>4s} {'prec':>7s} {'gain':>5s} {'valid':>6s}")
for cf in pit_cnxt_grid:
    for bf in pit_beta_grid:
        for pf in pit_pfloor_grid:
            f = agree_pit & (cnxt_pit_ood >= cf) & (beta_pit_ood >= bf) & (p_pit_ood >= pf) & (bidir_rol_ood < BIDIR_ROL_VETO)
            N = int(f.sum())
            if N == 0: continue
            bok = int(((beta_argmax_ood == ood_y) & f).sum())
            cok = int(((champ_pred_ood == ood_y) & f).sum())
            prec = bok / max(N, 1); g = bok - cok
            valid = (prec >= PIT_PREC_THR) and (g >= PIT_MIN_GAIN)
            r4_results.append({'cf': cf, 'bf': bf, 'pf': pf, 'N': N, 'bok': bok, 'cok': cok, 'prec': prec, 'gain': g, 'valid': valid})
            if valid:
                # Maximize gain; prefer LOWER cnxt_floor among valid train-OOD settings
                if (best_r4 is None) or (g > best_r4['gain']) or (g == best_r4['gain'] and cf < best_r4['cf']):
                    best_r4 = {'cf': cf, 'bf': bf, 'pf': pf, 'N': N, 'bok': bok, 'cok': cok, 'prec': prec, 'gain': g}
            if valid:
                print(f"  {cf:>8.3f} {bf:>8.3f} {pf:>8.2f} {N:>4d} {bok:>4d} {cok:>4d} {prec:>7.4f} {g:>+5d} {str(valid):>6s}")

if best_r4 is not None:
    R4_VALID = True
    PIT_CNXT_FLOOR = float(best_r4['cf']); PIT_BETA_FLOOR = float(best_r4['bf']); PIT_PFLOOR = float(best_r4['pf'])
    print(f"\n✅ R4 accepted: cnxt_floor={PIT_CNXT_FLOOR}, beta_pit_floor={PIT_BETA_FLOOR}, p_pit_floor={PIT_PFLOOR}, gain={best_r4['gain']}, prec={best_r4['prec']:.4f}")
else:
    R4_VALID = False
    PIT_CNXT_FLOOR = 1.01; PIT_BETA_FLOOR = 1.01; PIT_PFLOOR = 1.01
    best_r4 = {'N': 0, 'bok': 0, 'cok': 0, 'prec': 0.0, 'gain': 0}
    print(f"\n❌ R4: no (cf, bf, pf) combo passes prec>={PIT_PREC_THR} AND gain>={PIT_MIN_GAIN}.")